### Question 1: From Prices to Returns

In [11]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.api as sm
import warnings

warnings.filterwarnings("ignore")


# Load pre-processed data with date index
stock_excess = pd.read_csv("./processed_data/stock_excess_monthly.csv", index_col=0)
# Market excess and risk-free rate are single columns; read as Series aligned on the same date index
mkt_excess = pd.read_csv("./processed_data/mkt_excess_monthly.csv", index_col=0)[
    "mkt_excess"
]
rf_m = pd.read_csv("./processed_data/rf_monthly.csv", index_col=0)["rf"]

### Task (a)


In [ ]:
# Task (a): construct a monthly excess-return panel from the pre-processed data

# `stock_excess` already contains monthly stock excess returns (columns = tickers, index = date)
# `mkt_excess` is the corresponding monthly market excess return series.
# These tasks were completed in the data pre-processing file.

# Combine stock and market excess returns into a single DataFrame.
log_returns = stock_excess.copy()
log_returns["mkt_excess"] = mkt_excess

# Drop any months where the market excess return is missing.
log_returns = log_returns.dropna(subset=["mkt_excess"])

log_returns.head()

,A,AAPL,ABBV,ABT,ACGL,ACN,ADBE,ADI,ADM,ADP,...,WY,WYNN,XEL,XOM,XYL,YUM,ZBH,ZBRA,ZTS,mkt_excess
2015-03-31,-0.009829,-0.034014,-0.030641,-0.016088,0.035693,0.028604,-0.050217,0.074302,0.005257,-0.006663,...,-0.034745,-0.099366,0.014444,-0.026841,-0.020469,-0.018646,-0.012013,-0.004928,-0.000220,-0.016226
2015-04-30,-0.005942,0.004169,0.106092,0.005443,-0.016648,-0.002125,0.026668,-0.020185,0.029145,-0.014528,...,-0.052338,-0.126769,-0.027795,0.025900,0.053939,0.091635,-0.069184,0.013280,-0.041043,0.006884
2015-05-31,-0.006086,0.042616,0.027686,0.044168,0.049823,0.034211,0.037339,0.098645,0.081648,0.009683,...,0.031055,-0.095432,0.002395,-0.018639,-0.010135,0.045424,0.038199,0.172819,0.111997,0.008712
2015-06-30,-0.065001,-0.039978,0.006930,0.007788,0.044893,0.005636,0.021947,-0.059111,-0.093723,-0.059866,...,-0.026151,-0.022305,-0.048630,-0.025793,0.011540,-0.002373,-0.045571,0.010739,-0.033678,-0.023275
2015-07-31,0.057611,-0.035487,0.046502,0.035089,0.061636,0.061349,0.010019,-0.097737,-0.018736,-0.007756,...,-0.028057,0.043172,0.072524,-0.051151,-0.072986,-0.023631,-0.050398,-0.033287,0.013632,0.017544


### Task (b)

In [13]:
# Align the monthly risk-free rate (rf_m) to the excess-return panel.
# `rf_m` is a Series indexed by the same monthly dates.
rf_aligned = pd.DataFrame({"rf": rf_m.loc[log_returns.index]})

# Inspect the aligned risk-free rate.
rf_aligned.head()

,rf
2015-03-31,0.001517
2015-04-30,0.001600
2015-05-31,0.001725
2015-06-30,0.002039
2015-07-31,0.002006


In [14]:
# Check for missing values in the aligned monthly risk-free rate.
missing_rf = rf_aligned["rf"].isna().sum()
print(f"Missing rf values: {missing_rf}")

# Forward fill missing rf values if any.
rf_aligned["rf"] = rf_aligned["rf"].ffill()

print(f"Missing rf values after ffill: {rf_aligned['rf'].isna().sum()}")

Missing rf values: 0
Missing rf values after ffill: 0


In [15]:
# The series `rf` is already a monthly risk-free rate derived from DGS10.
# For later use we will keep it as-is and also compute its average.
mean_rf_monthly = rf_aligned["rf"].mean()

rf_aligned.head()

,rf
2015-03-31,0.001517
2015-04-30,0.001600
2015-05-31,0.001725
2015-06-30,0.002039
2015-07-31,0.002006


In [16]:
# Isolate only the stock excess-return columns for matrix operations.
returns_only = log_returns[stock_excess.columns]

# 1) Compute Annualized Expected Excess Returns (mu_hat)
# The columns in `returns_only` are monthly *excess* log returns that were
# constructed in `Data_Pre_Processing.ipynb` by first computing *daily* log
# returns, then summing them within each month and subtracting the monthly
# risk-free rate. For such monthly sums of daily log returns, we have the
# equivalence:
#   252 * (mean daily log return)  ==  12 * (mean monthly log return).
# Thus, multiplying the monthly excess-return mean by 12 produces the same
# annualized expected excess return as the formula in the assignment handout.
mu_hat = 12 * returns_only.mean()

# 2) Compute Annualized Covariance Matrix (sigma_hat)
# For the same reason, the annualized covariance matrix can be obtained as
# 12 times the covariance matrix of monthly excess returns, which is
# equivalent to 252 times the daily covariance matrix used upstream.
sigma_hat = 12 * returns_only.cov()

# Reporting
print(f"mu_hat shape: {mu_hat.shape}")
print(f"sigma_hat shape: {sigma_hat.shape}")
print("mu_hat (first 5):\n", mu_hat.head())
print("sigma_hat (top-left 5x5):\n", sigma_hat.iloc[:5, :5])

mu_hat shape: (472,)
sigma_hat shape: (472, 472)
mu_hat (first 5):
 A       0.090355
AAPL    0.184475
ABBV    0.139531
ABT     0.100968
ACGL    0.132852
dtype: float64
sigma_hat (top-left 5x5):
              A      AAPL      ABBV       ABT      ACGL
A     0.062311  0.026612  0.023499  0.030200  0.016727
AAPL  0.026612  0.073744  0.008053  0.022621  0.014229
ABBV  0.023499  0.008053  0.065017  0.019037  0.012242
ABT   0.030200  0.022621  0.019037  0.041945  0.005171
ACGL  0.016727  0.014229  0.012242  0.005171  0.060035


### Task (c)

In [17]:
# Annualized volatility (sqrt of diagonal of sigma_hat)
annualized_vol = pd.Series(np.sqrt(np.diag(sigma_hat)), index=sigma_hat.index)

# Find the top 2 and bottom 2 stocks, measured by annualized excess returns.
top2_returns = mu_hat.sort_values(ascending=False).head(2)
bottom2_returns = mu_hat.sort_values(ascending=True).head(2)

# Find the top 2 stocks, measured by annualized volatility.
top2_vol = annualized_vol.sort_values(ascending=False).head(2)

# Market correlation using the monthly market excess return.
market_col = "mkt_excess"
market_corr = returns_only.corrwith(log_returns[market_col])
# Find the stock with the lowest correlation with the market.
lowest_corr = market_corr.sort_values(ascending=True).head(1)

print("Top 2 annualized excess returns:")
print(top2_returns)
print("Bottom 2 annualized excess returns:")
print(bottom2_returns)
print("Top 2 annualized volatility:")
print(top2_vol)
print("Lowest correlation with market:")
print(lowest_corr)

# Summary table for outliers
outlier_tickers = pd.Index([])
outlier_tickers = outlier_tickers.union(top2_returns.index)
outlier_tickers = outlier_tickers.union(bottom2_returns.index)
outlier_tickers = outlier_tickers.union(top2_vol.index)
outlier_tickers = outlier_tickers.union(lowest_corr.index)

anomaly_summary = pd.DataFrame(
    {
        "Ticker": outlier_tickers,
        "Annualized Return": mu_hat.loc[outlier_tickers].values,
        "Annualized Volatility": annualized_vol.loc[outlier_tickers].values,
    }
).reset_index(drop=True)

anomaly_summary

Top 2 annualized excess returns:
NVDA    0.508029
AMD     0.322434
dtype: float64
Bottom 2 annualized excess returns:
WBA    -0.188733
VTRS   -0.185959
dtype: float64
Top 2 annualized volatility:
ENPH    0.833331
APA     0.818819
dtype: float64
Lowest correlation with market:
HRL    0.132161
dtype: float64


,Ticker,Annualized Return,Annualized Volatility
0,AMD,0.322434,0.546287
1,APA,-0.129088,0.818819
2,ENPH,0.119101,0.833331
3,HRL,-0.002413,0.202572
4,NVDA,0.508029,0.455064
5,VTRS,-0.185959,0.367281
6,WBA,-0.188733,0.303877
